# Transcript Theme-Relevance Analyzer — Colab / Kaggle runner

This notebook clones [transcript-theme-analyzer](https://github.com/Samuel-Effiong/transcript-theme-analyzer) — a public GitHub repo —
then installs dependencies, takes your API key, takes your transcripts, runs
the analysis, and shows the results. Since it clones instead of embedding
the code, re-running the clone cell always pulls whatever's currently on
GitHub — no manual re-syncing needed after you push local edits.

> **Before you run this: a compute-expectation note.** This pipeline is
> **I/O-bound** — every step is a network call to an LLM API (OpenAI or
> OpenRouter). There's no local heavy computation, so Colab/Kaggle's GPU/CPU
> power won't make it faster; the bottleneck is the LLM provider's response
> time and any rate limits, not your local machine. What this notebook *does*
> give you: a free hosted environment, zero local Python/venv setup, and
> convenient built-in secrets management for your API key.


## Get the code

In [ ]:
import os

REPO_DIR = "transcript-theme-analyzer"

if os.path.isdir(REPO_DIR):
    !cd {REPO_DIR} && git pull
else:
    !git clone https://github.com/Samuel-Effiong/transcript-theme-analyzer.git

%cd {REPO_DIR}


## Install dependencies

In [ ]:
!pip install -q -U "gdown>=6.0.0" "openai>=1.50.0" "pydantic>=2.6.0" "python-docx>=1.1.0" tqdm

import gdown
print(f"gdown version in use: {gdown.__version__}")
if tuple(int(x) for x in gdown.__version__.split(".")[:3]) < (6, 0, 0):
    print(
        "WARNING: gdown is still below 6.0.0 despite the upgrade above. "
        "This usually means gdown was already imported earlier in this same "
        "runtime, so Python is reusing the old module from memory. "
        "Restart the runtime (Colab: Runtime -> Restart session; "
        "Kaggle: Run -> Restart session) and re-run all cells from the top."
    )


## Configure your LLM provider + API key

This cell auto-detects where it's running:
- **Colab** — reads from Colab's Secrets manager (the key icon in the left
  sidebar). Add a secret named `OPENROUTER_API_KEY` there first, and make
  sure "Notebook access" is toggled on for it.
- **Kaggle** — reads from Kaggle's Secrets add-on (Add-ons → Secrets). Add a
  secret named `OPENROUTER_API_KEY` there first.
- **Neither found** — falls back to a masked prompt (`getpass`) so it still
  works, just not persisted anywhere.

This is set up to call **DeepSeek via OpenRouter**, which is far cheaper than
frontier models for this workload — roughly 30x less than Claude Sonnet — and
supports native structured output, so responses parse reliably.

Get a key at [openrouter.ai/keys](https://openrouter.ai/keys) and add credits at
[openrouter.ai/settings/credits](https://openrouter.ai/settings/credits).
OpenRouter is prepaid: with no balance every call fails with a 402.

Model options (set `LLM_DEFAULT_MODEL` below, or `MODELS` in the run cell):

| Model | $/1M in | $/1M out | Notes |
|---|---|---|---|
| `deepseek/deepseek-v4-flash` | $0.083 | $0.165 | default; 1M context |
| `deepseek/deepseek-v4-flash-0731` | $0.06 | $0.12 | cheapest; pinned snapshot |
| `deepseek/deepseek-v3.2` | $0.26 | $0.38 | 164K context |
| `deepseek/deepseek-v4-pro` | $0.87 | $1.74 | strongest |

Prefer a pinned snapshot for a long batch: a floating alias can be re-pointed
mid-run, which would leave half your corpus scored by a different model.

Switching providers (OpenAI, Anthropic direct) needs no code change — see
`.env.example` in the repo for the equivalent `LLM_*` settings.


In [ ]:
import getpass
import os


def set_secret_env(env_var: str, prompt: str | None = None) -> None:
    if os.environ.get(env_var):
        return

    value = None
    try:
        from google.colab import userdata  # type: ignore
        value = userdata.get(env_var)
    except Exception:
        pass

    if not value:
        try:
            from kaggle_secrets import UserSecretsClient  # type: ignore
            value = UserSecretsClient().get_secret(env_var)
        except Exception:
            pass

    if not value:
        value = getpass.getpass(prompt or f"Enter {env_var}: ")

    os.environ[env_var] = value


set_secret_env("OPENROUTER_API_KEY")

os.environ.setdefault("LLM_PROVIDER", "openrouter")
os.environ.setdefault("LLM_BASE_URL", "https://openrouter.ai/api/v1")
os.environ.setdefault("LLM_DEFAULT_MODEL", "deepseek/deepseek-v4-flash")
os.environ.setdefault("OPENROUTER_APP_NAME", "transcript-theme-analyzer")

os.environ.setdefault("CHUNK_SIZE_TOKENS", "12000")
os.environ.setdefault("CHUNK_OVERLAP_TOKENS", "800")
os.environ.setdefault("SINGLE_PASS_TOKEN_LIMIT", "20000")
os.environ.setdefault("MAX_CONCURRENT_CHUNKS", "8")
os.environ.setdefault("MAX_CONCURRENT_TRANSCRIPTS", "4")
os.environ.setdefault("MAX_CONCURRENT_REQUESTS", "12")
os.environ.setdefault("LLM_MAX_RETRIES", "5")
os.environ.setdefault("LLM_REQUEST_TIMEOUT", "180")
os.environ.setdefault("LLM_MAX_OUTPUT_TOKENS", "10000")

print("LLM_PROVIDER:", os.environ["LLM_PROVIDER"])
print("LLM_DEFAULT_MODEL:", os.environ["LLM_DEFAULT_MODEL"])
print("API key set:", bool(os.environ.get("OPENROUTER_API_KEY")))


## ⚠️ Kaggle only: enable internet access

Kaggle notebooks have internet access **off by default**. Since every
analysis call needs to reach the LLM API (and `gdown` downloads from Google Drive),
you must turn it on before running: **Settings (right sidebar) → Internet → On**
(requires phone verification on your Kaggle account, one-time).
Colab has internet on by default — nothing to do there.


## Get your transcripts — Option 1: Google Drive

Two ways to pull transcripts from Drive, depending on where you're running:

**Colab (recommended) — mount your own Drive.** This reads files directly
off an authenticated filesystem instead of downloading through Drive's
anonymous public-link endpoint, so it's not subject to that endpoint's
"too many recent accesses" throttling, and there's no 50-files-per-folder
limit. It also scales to any number of people without anyone sharing login
credentials — each person does this once, on their own account:

1. Get the folder shared to your Google account (or an "anyone with the
   link" link).
2. In Google Drive's web UI, right-click the folder → **"Add shortcut to
   Drive"** (sometimes labeled "Organize → Add shortcut"). This adds it to
   *your own* "My Drive" — the "Shared with me" view alone isn't enough,
   Colab's mount only sees "My Drive."
3. Set `GDRIVE_MOUNT_SUBPATH` in the next cell to wherever you placed the
   shortcut (e.g. `"MyDrive/GLH Transcripts"`), then run that cell — it'll
   prompt you to authorize Colab's access to your Drive.

**Kaggle, or Colab without a shortcut set up — shared folder link download.**
Paste your shared Google Drive folder URL (or folder ID) into
`GDRIVE_FOLDER_URL` in the cell after that. Make sure the folder's sharing
setting is **"Anyone with the link can view."** `gdown` will download the
folder and all its subfolders into `transcripts/`. Either way, the analyzer
automatically traverses all nested subfolders to find every `.txt` and
`.docx` transcript file.
</cell id="cc40f0af">

In [ ]:
import os

TRANSCRIPT_DIR = "transcripts"
USING_DRIVE_MOUNT = False

# Colab only: path under /content/drive to your transcripts folder or shortcut.
# See the markdown cell above for how to set this up (one-time, per person).
GDRIVE_MOUNT_SUBPATH = ""  # e.g. "MyDrive/GLH Transcripts"

try:
    from google.colab import drive  # type: ignore
    is_colab = True
except ImportError:
    is_colab = False

if is_colab and GDRIVE_MOUNT_SUBPATH.strip():
    try:
        drive.mount("/content/drive")
    except Exception as exc:
        print(f"Drive mount failed or was cancelled: {exc}\nFalling back to the download cell below.")
    else:
        mounted_path = os.path.join("/content/drive", GDRIVE_MOUNT_SUBPATH.strip())
        if os.path.isdir(mounted_path):
            TRANSCRIPT_DIR = mounted_path
            USING_DRIVE_MOUNT = True
            print(f"Using mounted Google Drive folder: {TRANSCRIPT_DIR}")
        else:
            print(
                f"Mounted Drive, but couldn't find a folder at {mounted_path}. "
                "Check that GDRIVE_MOUNT_SUBPATH matches where you added the shortcut "
                "in your Drive. Falling back to the download cell below."
            )
elif is_colab:
    print("On Colab but GDRIVE_MOUNT_SUBPATH is empty -- skipping Drive mount, "
          "falling back to the download cell below.")
else:
    print("Not running on Colab -- Drive mount isn't available here (e.g. on Kaggle), "
          "falling back to the download cell below.")


In [ ]:
import concurrent.futures
import os
import threading
import time

import gdown
from tqdm.auto import tqdm

MAX_WORKERS = 6  # concurrent downloads; lower this if Drive starts throwing more "too many accesses" errors

# Paste your shared Google Drive folder URL or ID here:
GDRIVE_FOLDER_URL = ""  # e.g. "https://drive.google.com/drive/folders/1abc..." or "1abc..."

if USING_DRIVE_MOUNT:
    print(f"Already using transcripts from mounted Drive at {TRANSCRIPT_DIR} -- skipping download.")
elif GDRIVE_FOLDER_URL.strip():
    if tuple(int(x) for x in gdown.__version__.split(".")[:3]) < (6, 0, 0):
        raise RuntimeError(
            f"gdown {gdown.__version__} is loaded, but folders with more than 50 files "
            "need gdown>=6.0.0. Re-run the 'Install dependencies' cell above, then if the "
            "version still isn't >=6.0.0, restart the runtime (Colab: Runtime -> Restart "
            "session; Kaggle: Run -> Restart session) and re-run all cells from the top -- "
            "an old gdown already imported this session won't be replaced by pip alone."
        )

    os.makedirs(TRANSCRIPT_DIR, exist_ok=True)

    print("Listing files in the shared Google Drive folder (this can take a moment for large folders)...")
    list_start = time.monotonic()
    try:
        file_list = gdown.download_folder(
            url=GDRIVE_FOLDER_URL.strip(),
            output=TRANSCRIPT_DIR,
            quiet=False,
            skip_download=True,
        )
    except Exception as exc:
        file_list = []
        print(f"Could not list folder contents: {exc}")
    print(f"Listing done in {time.monotonic() - list_start:.1f}s.\n")

    total = len(file_list)
    print(f"Found {total} file(s). Downloading into {TRANSCRIPT_DIR}/ with {MAX_WORKERS} parallel workers...")
    print("(If you interrupt this cell, up to MAX_WORKERS downloads already in flight will "
          "still finish in the background -- a running network call can't be aborted from "
          "outside. Everything else queued stops immediately. For a full, instant stop, "
          "restart the runtime instead.)\n")

    succeeded, failed = [], []
    lock = threading.Lock()

    def _download_one(gfile):
        os.makedirs(os.path.dirname(gfile.local_path), exist_ok=True)
        gdown.download(
            url="https://drive.google.com/uc?id=" + gfile.id,
            output=gfile.local_path,
            quiet=True,
        )

    start_time = time.monotonic()
    pool = concurrent.futures.ThreadPoolExecutor(max_workers=MAX_WORKERS)
    try:
        with tqdm(total=total, unit="file", desc="Downloading") as pbar:
            future_to_file = {pool.submit(_download_one, gfile): gfile for gfile in file_list}
            for future in concurrent.futures.as_completed(future_to_file):
                gfile = future_to_file[future]
                exc = future.exception()
                with lock:
                    if exc is None:
                        succeeded.append(gfile.path)
                    else:
                        reason = str(exc).strip().splitlines()[0] if str(exc).strip() else type(exc).__name__
                        failed.append((gfile.path, gfile.id, reason))
                        tqdm.write(f"FAIL  {gfile.path}  ({reason})")
                    pbar.set_postfix(ok=len(succeeded), failed=len(failed))
                pbar.update(1)
        pool.shutdown(wait=True)
    except KeyboardInterrupt:
        pool.shutdown(wait=False, cancel_futures=True)
        print(f"\nInterrupted: cancelled all queued downloads. Up to {MAX_WORKERS} already "
              f"in-flight may still finish writing in the background.")
        raise

    total_elapsed = time.monotonic() - start_time
    print(f"\nDownload finished in {total_elapsed:.1f}s: {len(succeeded)} succeeded, {len(failed)} failed out of {total}.")

    if failed:
        print("\nFailed downloads (likely need 'Anyone with the link' sharing enabled):")
        for path, file_id, reason in failed:
            print(f"  - {path}")
            print(f"      id:     {file_id}")
            print(f"      url:    https://drive.google.com/file/d/{file_id}/view")
            print(f"      reason: {reason}")
else:
    print("GDRIVE_FOLDER_URL is empty. Skip if using local upload or Kaggle dataset fallback below.")


## Option 2: Fallbacks (Kaggle Dataset/Zip Upload or Colab Manual Upload)

If you didn't use Option 1 above:
- **Kaggle, already-extracted Dataset**: Attach a dataset via "Add Input", set `KAGGLE_TRANSCRIPT_DIR` below.
- **Kaggle, zip file**: Upload your transcripts as a single `.zip` (via "Add Input" → create/attach a
  Dataset containing that zip — Kaggle Datasets are the only way to get files into a Kaggle notebook).
  Set `KAGGLE_TRANSCRIPT_ZIP` below to the zip's path under `/kaggle/input/...`, or leave it blank and
  the cell will auto-find the first `.zip` under `/kaggle/input`. The cell extracts it into
  `transcripts/`, preserving whatever folder structure is inside the zip.
- **Colab Upload**: Use Colab's interactive file upload — pick individual `.txt`/`.docx` files, or a
  single `.zip`, which gets auto-extracted the same way.
</cell id="e808bba7">

In [ ]:
import glob
import os
import zipfile


def extract_zip(zip_path, dest_dir):
    os.makedirs(dest_dir, exist_ok=True)
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(dest_dir)
    print(f"Extracted {zip_path} into {dest_dir}/")


# Check if we have files in TRANSCRIPT_DIR already
has_gdrive_files = os.path.isdir(TRANSCRIPT_DIR) and len(os.listdir(TRANSCRIPT_DIR)) > 0

if not has_gdrive_files:
    if os.path.isdir("/kaggle/input"):
        KAGGLE_TRANSCRIPT_DIR = "/kaggle/input/datasets/samuelnkopuruk/transcripts-raw"  # <-- change if needed
        KAGGLE_TRANSCRIPT_ZIP = ""  # <-- set to a specific .zip under /kaggle/input, or leave blank to auto-find one

        if os.path.isdir(KAGGLE_TRANSCRIPT_DIR):
            TRANSCRIPT_DIR = KAGGLE_TRANSCRIPT_DIR
            print(f"Running on Kaggle — using input dataset at {TRANSCRIPT_DIR}")
        else:
            zip_path = KAGGLE_TRANSCRIPT_ZIP.strip()
            if not zip_path:
                found_zips = sorted(glob.glob("/kaggle/input/**/*.zip", recursive=True))
                if found_zips:
                    zip_path = found_zips[0]
                    print(f"KAGGLE_TRANSCRIPT_ZIP not set — auto-found {zip_path}")
                    if len(found_zips) > 1:
                        print(f"(Note: {len(found_zips)} zip files found under /kaggle/input; "
                              f"using the first one. Set KAGGLE_TRANSCRIPT_ZIP explicitly to pick a different one.)")

            if zip_path and os.path.isfile(zip_path):
                extract_zip(zip_path, TRANSCRIPT_DIR)
            else:
                print("No extracted dataset directory or .zip found under /kaggle/input. "
                      "Attach a Dataset via 'Add Input' and set KAGGLE_TRANSCRIPT_DIR or KAGGLE_TRANSCRIPT_ZIP above.")
    else:
        try:
            from google.colab import files  # type: ignore
            os.makedirs(TRANSCRIPT_DIR, exist_ok=True)
            print(f"Upload your .txt/.docx transcript files, or a single .zip, into {TRANSCRIPT_DIR}/:")
            uploaded = files.upload()
            for name, content in uploaded.items():
                saved_path = os.path.join(TRANSCRIPT_DIR, name)
                with open(saved_path, "wb") as f:
                    f.write(content)
                if name.lower().endswith(".zip"):
                    extract_zip(saved_path, TRANSCRIPT_DIR)
                    os.remove(saved_path)
            print(f"Uploaded {len(uploaded)} file(s)/archive(s) into {TRANSCRIPT_DIR}/")
        except ImportError:
            pass

print(f"Final TRANSCRIPT_DIR set to: {TRANSCRIPT_DIR}")


In [ ]:
from transcript_theme_analyzer.cli import DEFAULT_GLOB_PATTERNS, _discover_transcript_paths

_found = _discover_transcript_paths(TRANSCRIPT_DIR, None)
print(f"Found {len(_found)} transcript file(s) in {TRANSCRIPT_DIR} "
      f"(matching {' or '.join(DEFAULT_GLOB_PATTERNS)}, including subfolders):")
for _p in _found:
    print(f"  - {_p}")
if not _found:
    print("  (none — check TRANSCRIPT_DIR above before running the analysis cell)")

## Run the analysis

Edit `THEME` and `MODELS` below, then run this cell. It calls the same
`transcript_theme_analyzer.cli` you'd run locally — see `.env.example` in the
repo for the full list of tuning knobs.

**This cell resumes.** Every transcript is checkpointed to
`OUT_DIR/.checkpoint.jsonl` the moment it finishes. If the runtime times out,
the tab closes, or you interrupt it, just run this cell again — it skips
everything already done and picks up the remainder. Nothing is paid for twice.
Changing `THEME` or `MODELS` correctly invalidates the checkpoint and starts over.

**Start small.** Set `LIMIT` to something like 20 for a trial run, check the
report, and only then set it back to `None` for the full corpus. A large batch
is a multi-hour, many-million-token job; it's worth confirming the theme gives
you what you want first.

**If you see rate-limit warnings**, lower `MAX_CONCURRENT_REQUESTS`. Progress
is reported per chunk and per transcript, with a running ETA.


In [ ]:
THEME = "the glory of God"           # <-- change this to your search theme
MODELS = "deepseek/deepseek-v4-flash"  # space-separated if running more than one model
OUT_DIR = "results"

LIMIT = 20           # analyze only the first N transcripts; set to None for the whole corpus
MAX_TRANSCRIPTS = 4  # transcripts analyzed at once
MAX_REQUESTS = 12    # global cap on simultaneous API calls -- lower if you hit rate limits

# Assembled as a single string rather than a multi-line `!` command, because
# backslash continuation inside `!` is not reliable across IPython versions.
# -u forces unbuffered output so progress lines appear as the run happens,
# rather than all at once when the cell finishes.
CMD = (
    f'python -u -m transcript_theme_analyzer.cli'
    f' --transcript-dir "{TRANSCRIPT_DIR}"'
    f' --theme "{THEME}"'
    f' --models {MODELS}'
    f' --out-dir "{OUT_DIR}"'
    f' --max-concurrent-transcripts {MAX_TRANSCRIPTS}'
    f' --max-concurrent-requests {MAX_REQUESTS}'
    + (f' --limit {LIMIT}' if LIMIT else '')
)
print(CMD, "\n")

!{CMD}

## View and download the results

Two final outputs land in `OUT_DIR`: `report.html` (rendered inline below)
and `report.docx`. On Colab, `report.docx` downloads directly. On Kaggle,
everything under `/kaggle/working` is automatically saved as notebook
output and downloadable from the Output tab after you save a version.


In [ ]:
import os

from IPython.display import HTML, display

_report = f"{OUT_DIR}/report.html"
_size_mb = os.path.getsize(_report) / 1e6

# Rendering a very large report inline will lock up (or crash) the browser
# tab -- a full corpus can produce tens of MB of HTML. Past a safe threshold,
# point at the file instead of embedding it.
if _size_mb > 5:
    print(f"report.html is {_size_mb:.1f} MB — too large to render inline without hanging the tab.")
    print("Download it instead and open it locally:")
    print(f"  - Colab: file browser on the left -> {_report} -> Download")
    print(f"  - Kaggle: Output tab -> {_report}")
    try:
        from google.colab import files  # type: ignore
        files.download(_report)
    except ImportError:
        pass
else:
    with open(_report, encoding="utf-8") as f:
        display(HTML(f.read()))


In [ ]:
try:
    from google.colab import files  # type: ignore
    files.download(f"{OUT_DIR}/report.docx")
except ImportError:
    print(f"Not on Colab — find {OUT_DIR}/report.docx in the Output tab (Kaggle) "
          f"or the file browser on the left (other environments).")
